In [ ]:
# Customer Churn Prediction — Company-Level ML Pipeline

# =====================================
# ******* IMPORT LIBRARIES *******
# =====================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# ==================================
# ******* LOAD DATASET *******
# ==================================

# Replace with your file name
df = pd.read_csv(r"C:\Users\priya\Downloads\Telco-Customer-Churn.csv")

# ==================================
# ******* BASIC CLEANING *******
# ==================================

# Remove customerID
df.drop("customerID", axis=1, inplace=True)

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors='coerce'
)

# Convert target column
df["Churn"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

# =====================================
# ******* FEATURES & TARGET *******
# =====================================
X = df.drop("Churn", axis=1)
y = df["Churn"]

# =====================================
# ******* TRAIN TEST SPLIT *******
# =====================================
Xtrain, Xtest, ytrain, ytest = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ====================================================
# ******* NUMERICAL & CATEGORICAL COLUMNS *******
# ====================================================
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# =======================================
# ******* NUMERICAL PIPELINE *******
# =======================================
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# =========================================
# ******* CATEGORICAL PIPELINE *******
# =========================================
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

# ===================================
# ******* PREPROCESSOR *******
# ===================================
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# =================================================
# ******* LOGISTIC REGRESSION PIPELINE *******
# =================================================
logistic_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        C=0.5,
        solver='liblinear'
    ))
])

# ==================================
# ******* TRAIN MODEL *******
# ==================================
logistic_pipeline.fit(Xtrain, ytrain)

# ==================================
# ******* PREDICTION *******
# ==================================
y_pred = logistic_pipeline.predict(Xtest)
y_prob = logistic_pipeline.predict_proba(Xtest)[:, 1]

# ==============================
# ******* EVALUATION *******
# ==============================
print("="*50)
print("LOGISTIC REGRESSION RESULTS")
print("="*50)

print("Accuracy:", accuracy_score(ytest, y_pred))
print("\nClassification Report:\n")
print(classification_report(ytest, y_pred))
print("\nConfusion Matrix:\n")
print(confusion_matrix(ytest, y_pred))
print("\nROC-AUC Score:", roc_auc_score(ytest, y_prob))

# ============================================
# ******* RANDOM FOREST PIPELINE *******
# ============================================
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        random_state=42,
        class_weight='balanced'
    ))
])

# =========================================
# ******* TRAIN RANDOM FOREST *******
# =========================================
rf_pipeline.fit(Xtrain, ytrain)

# =========================================
# ******* RANDOM FOREST PREDICTION *******
# =========================================
rf_pred = rf_pipeline.predict(Xtest)
rf_prob = rf_pipeline.predict_proba(Xtest)[:, 1]

# ============================================
# ******* RANDOM FOREST RESULTS *******
# ============================================ 
print("\n" + "="*50)
print("RANDOM FOREST RESULTS")
print("="*50)
print("Accuracy:", accuracy_score(ytest, rf_pred))
print("\nClassification Report:\n")
print(classification_report(ytest, rf_pred))
print("\nConfusion Matrix:\n")
print(confusion_matrix(ytest, rf_pred))
print("\nROC-AUC Score:", roc_auc_score(ytest, rf_prob))

# ========================================
# ******* SAVE BEST MODEL *******
# ========================================
import pickle

pickle.dump(rf_pipeline, open("best_model.pkl", "wb"))

print("\nModel Saved Successfully")

C:\Users\priya\AppData\Local\Temp\ipykernel_7128\3550339103.py:91: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object']).columns


LOGISTIC REGRESSION RESULTS
Accuracy: 0.7381121362668559

Classification Report:

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.50      0.79      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409


Confusion Matrix:

[[746 289]
 [ 80 294]]

ROC-AUC Score: 0.8418145650882224

RANDOM FOREST RESULTS
Accuracy: 0.7679205110007097

Classification Report:

              precision    recall  f1-score   support

           0       0.89      0.78      0.83      1035
           1       0.55      0.73      0.63       374

    accuracy                           0.77      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.77      0.78      1409


Confusion Matrix:

[[808 227]
 [100 274]]

ROC-AUC Score: 0.8412462218088818

Model Saved Successfully
